In [ ]:
import sys
sys.path.append(r'C:\Users\sadiq\Desktop\U_Vienna\codes\icm-zebrafish')
from scipy.stats import zscore
import numpy as np
import matplotlib.pyplot as plt
from ica_utils import *
%matplotlib inline

In [ ]:
%reload_ext autoreload
%autoreload 2
from ica_utils import *

In [ ]:
traces = np.load('fluo_plane100_160.npy')
time_idx = np.load('fluo_plane100_160_time_indices.npy')
tail_angle= np.load('220210_F2_run5_tail_angle.npy')

In [ ]:
f_s = 5.5573

In [ ]:
traces.shape

In [ ]:
n = eig_dec(traces)
# n = traces.shape[0]
# len(tail_angle)

In [ ]:
plt.figure(figsize=(20,2))
plt.plot(tail_angle)

In [ ]:
# resampling behavior to same number of samples as the traces

from scipy.signal import resample
aa = resample(tail_angle,3333,domain='time')
plt.figure(figsize=(20,2))
plt.plot(aa)

In [ ]:
ic_comps,IC_ft,A,mean = ica_dec(traces,n,t=0.001,max_=500)

In [ ]:
plt.imshow(np.corrcoef(ic_comps.T))

In [ ]:
# stop

In [ ]:
IC_ft.shape

In [ ]:
def plott_ics(ics):        
    fig,ax = plt.subplots(ics.shape[1],1,figsize=(15,1.5*ics.shape[1]))
    for i in range(ics.shape[1]):
        ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=ics.T[i,:].min(),ymax=ics.T[i,:].max(),ls='--',color='g',lw = .7)
        ax[i].plot((ics.T[i,:]),label='{}'.format(i))
        ax[i].legend()

In [ ]:
%matplotlib inline
plott_ics(ic_comps)

In [ ]:
plot_FT_spectrals(ic_comps,f_s,n)

In [ ]:
plt.plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2, 3333)),IC_ft.T)
plt.xlim(0,3)

In [ ]:
new_mat, predictions= cluster(ic_comps,2,f_s)

In [ ]:
np.sum(predictions==0)

In [ ]:
al = np.c_[new_mat,predictions,np.arange(n)]
al.round(3)

In [ ]:
%matplotlib notebook
plot_clusters(new_mat,predictions)   # 1 are noise, 0 are signals

In [ ]:
group0 = IC_ft[np.where(al[:,3] == 1)]
group0.shape

In [ ]:
plottings_spectrals(IC_ft,2,al,f_s)

In [ ]:
plottings_logSpectral(IC_ft,2,al,f_s)

In [ ]:
# indexes to set to zeros
idx_IC_Clust_pred = np.where(predictions==1)
idx_IC_Clust_pred

In [ ]:
new_ic_comps= ic_comps.copy()
new_ic_comps[:,idx_IC_Clust_pred] = 0

In [ ]:
# cleaning F1_T1 
cleaned_traces = np.dot(new_ic_comps, A.T) + mean

In [ ]:
cleaned_traces.shape

In [ ]:
%matplotlib inline
fig,ax = plt.subplots(cleaned_traces.shape[1],1,figsize=(15,1.2*cleaned_traces.shape[1]))
for i in range(cleaned_traces.shape[1]):
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',lw=.75)
    ax[i].plot(cleaned_traces[:,i]+.75,label=f'clean{i}',color='blue',alpha=0.85,lw=.75)
    ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),lw=0.7,ls='--',color='g')
    ax[i].legend()

In [ ]:
# zoomed of selected ones cleaned traces 
%matplotlib inline
fig,ax = plt.subplots(cleaned_traces.shape[1],1,figsize=(15,1.2*cleaned_traces.shape[1]))
for i in range(cleaned_traces.shape[1]):
    ax[i-6].plot(traces[i,:],label=f'original{i}',color='red')
    ax[i-6].plot(cleaned_traces[:,i]+.75,label=f'cleaned{i}',color='blue',alpha=0.85)
    ax[i-6].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g',lw =.7)
    ax[i-6].set_xlim([1800,2000])
    ax[i-6].legend()

#### Three Clusters

In [ ]:
new_mat3, predictions3= cluster(ic_comps,3,f_s)

In [ ]:
%matplotlib notebook
plot_clusters(new_mat3,predictions3)

In [ ]:
al3 = np.c_[new_mat3.round(2),predictions3.round(1),np.arange(n)]
al3

In [ ]:
def plottings_spectrals3(n_clus,alll):
    fig,ax=plt.subplots(1,n_clus,figsize=(15,3))
    for i in range(n_clus):
        group = IC_ft[np.where(alll[:,3] == i)]
        for j in range(group.shape[0]):
            ax[i].plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3333)),group[j,:])
        ax[i].plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3333)),group.mean(0),color='black',lw=1,label='mean')
        ax[i].set_xlim([0,1])
        ax[i].set_ylim([0,5])
        ax[i].set_title('cl {} with {}'.format(i,group.shape[0]))
        ax[i].legend()

def plottings_logSpectral3(n_clus,alll):
    plt.figure(figsize=(15,8))
    for i in range(n_clus):
        group = IC_ft[np.where(alll[:,3] == i)]
        plt.plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3333)),np.log(group.mean(0)),lw=1,label='{}'.format(i))
        plt.xlim([0,.5])
        plt.ylim([0,2.5])
        plt.legend()

In [ ]:
plottings_logSpectral3(3,al3)

In [ ]:
plottings_spectrals3(3,al3)

In [ ]:
%matplotlib inline
unique,count = np.unique(predictions3,return_counts=True)
plt.stem(unique,count)
plt.title('{} ICs'.format(np.sum(count)))
plt.xlabel('clusters')
plt.ylabel('count')
plt.grid()

In [ ]:
# 0 alone
%matplotlib inline
order = [0,2,1]
ic_ = ic_comps.copy()
idx = np.where(predictions3!=order[0])
ic_[:,idx[0]] = 0
cleaned = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned.shape[1],1,figsize=(15,1.2*cleaned.shape[1]))
for i in range(cleaned.shape[1]):
    ax[i].plot(cleaned[:,i]+.75,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),lw=.7,ls='--',color='g')
    ax[i].legend()

In [ ]:
### for 0 and 2
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions3==0)[0],np.where(predictions3==2)[0]])
ic_[:,new_idx] = 0
cleaned = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned.shape[1],1,figsize=(15,1.2*cleaned.shape[1]))
for i in range(cleaned.shape[1]):
    ax[i].plot(cleaned[:,i]+.75,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),lw=.7,ls='--',color='g')
    ax[i].legend()

#### Seven clusters

In [ ]:
new_mat7, predictions7= cluster(ic_comps,7,f_s)

In [ ]:
%matplotlib notebook
plot_clusters(new_mat7,predictions7)

In [ ]:
al7 = np.c_[new_mat7.round(2),predictions7.round(1),np.arange(n)]
al7

In [ ]:
plottings_logSpectral3(7,al7)

In [ ]:
plottings_spectrals3(7,al7)

In [ ]:
%matplotlib inline
unique,count = np.unique(predictions7,return_counts=True)
plt.stem(unique,count)
plt.title('{} ICs'.format(np.sum(count)))
plt.xlabel('clusters')
plt.ylabel('count')
plt.grid()

In [ ]:
order = [6,1,2,3,4,5,0]

In [ ]:
### for  2, and 1
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==1)[0],np.where(predictions7==2)[0]])
ic_[:,new_idx] = 0
cleaned = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned.shape[1],1,figsize=(15,1.2*cleaned.shape[1]))
for i in range(cleaned.shape[1]):
    ax[i].plot(cleaned[:,i]+3,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),lw=.7,ls='--',color='g')
    ax[i].legend()

In [ ]:
plt.figure(figsize=(15,1.2))
plt.plot(cleaned[:,10],label=f'cleaned{i}',color='blue',lw=.7)
plt.plot(traces[10,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)

In [ ]:
### for 2, 1 and 3
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==1)[0],np.where(predictions7==2)[0],np.where(predictions7==3)[0]])
ic_[:,new_idx] = 0
cleaned = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned.shape[1],1,figsize=(15,1.2*cleaned.shape[1]))
for i in range(cleaned.shape[1]):
    ax[i].plot(cleaned[:,i]+3,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),lw=.7,ls='--',color='g')
    ax[i].legend()

In [ ]:
### for 6, 1, 2, and 3
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==1)[0],np.where(predictions7==2)[0],np.where(predictions7==6)[0],np.where(predictions7==3)[0]])
ic_[:,new_idx] = 0
cleaned = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned.shape[1],1,figsize=(15,1.2*cleaned.shape[1]))
for i in range(cleaned.shape[1]):
    ax[i].plot(cleaned[:,i]+3,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),lw=.7,ls='--',color='g')
    ax[i].legend()

In [ ]:
np.random.seed(42)
a= np.random.rand(36).reshape((6,6))
print(a)
n_neur=2

In [ ]:
a[:n_neur,n_neur:n_neur*2]

In [ ]:
jj= ic_comps.copy()

